# Reproducible quantitative analysis

**Manuscript:** *Quantifying transient biochemical information in Escherichia coli chemotaxis*

This notebook runs the same master pipeline as `chemotaxis_reproducible_analysis.py`.

### Important scope note
The notebook recomputes the static and calibration results from source/model inputs.  
The final open-domain dynamic methylation-rate medians and Monte Carlo intervals are retained as the manuscript's validated final outputs because the original trajectory/spectrum simulation layer is not embedded in this package. This is stated explicitly to avoid silently inventing a replacement model.


In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import chemotaxis_reproducible_analysis as cra


## 1. Run the complete validated pipeline

In [2]:
summary = cra.main()
summary



=== PRINCIPAL REPRODUCIBLE RESULTS ===
{
  "H_micro_bits_per_cell": 60000.0,
  "H_count_bits_per_cell": 15.872698924987581,
  "alpha": 1.9946442705633443,
  "m0": 0.4365445164420451,
  "GmA": 2.659525694084459,
  "alpha_within_state_95": [
    1.9478385541503485,
    2.0569372081288204
  ],
  "alpha_structural_95": [
    1.841463993196766,
    2.155739297587945
  ],
  "sigma_m_anchors": [
    0.2550202841886844,
    0.15228279271782752,
    0.13649050310264538
  ],
  "I_bits": 2.181632308931791,
  "N_eff": 4.536665559535445,
  "I_prior_sensitivity_min": 1.922677425403308,
  "I_prior_sensitivity_max": 2.2574349841927455,
  "N_eff_prior_sensitivity_min": 3.791260071559628,
  "N_eff_prior_sensitivity_max": 4.781406234667399,
  "tau_env_min_range_min": [
    4.229584230195842,
    16.918336920783368
  ],
  "epsilon_ad_range": [
    0.009851244093733861,
    0.039404976374935445
  ],
  "corner_frequency_Hz": 0.015915494309189534,
  "dynamic_open_rates_bits_s": [
    0.000174,
    0.000667,

{'H_micro_bits_per_cell': 60000.0,
 'H_count_bits_per_cell': 15.872698924987581,
 'alpha': 1.9946442705633443,
 'm0': 0.4365445164420451,
 'GmA': 2.659525694084459,
 'alpha_within_state_95': [1.9478385541503485, 2.0569372081288204],
 'alpha_structural_95': [1.841463993196766, 2.155739297587945],
 'sigma_m_anchors': [0.2550202841886844,
  0.15228279271782752,
  0.13649050310264538],
 'I_bits': 2.181632308931791,
 'N_eff': 4.536665559535445,
 'I_prior_sensitivity_min': 1.922677425403308,
 'I_prior_sensitivity_max': 2.2574349841927455,
 'N_eff_prior_sensitivity_min': 3.791260071559628,
 'N_eff_prior_sensitivity_max': 4.781406234667399,
 'tau_env_min_range_min': [4.229584230195842, 16.918336920783368],
 'epsilon_ad_range': [0.009851244093733861, 0.039404976374935445],
 'corner_frequency_Hz': 0.015915494309189534,
 'dynamic_open_rates_bits_s': [0.000174, 0.000667, 0.00142, 0.0024],
 'kinase_rates_bits_s': [0.0022000000000000006,
  0.008800000000000002,
  0.019799999999999998,
  0.0352000000

## 2. Capacity hierarchy

In [3]:
H_micro, H_count = cra.formal_capacities()
print(f"Microscopic site-resolved capacity: {H_micro:,.0f} bits/cell")
print(f"Total-count capacity: {H_count:.3f} bits/cell")


Microscopic site-resolved capacity: 60,000 bits/cell
Total-count capacity: 15.873 bits/cell


## 3. Shimizu endpoint calibration and uncertainty layers

In [4]:
states, scale = cra.load_shimizu_source()
fm_df = cra.fit_all_fm(states, scale)
cal = cra.endpoint_calibration(fm_df)
print("alpha =", cal["alpha"])
print("m0 =", cal["m0"])
print("GmA =", cal["GmA"])
fm_df


alpha = 1.9946442705633443
m0 = 0.4365445164420451
GmA = 2.659525694084459


,state,fm,mse_activity,n_points
0,QEEE,0.333677,0.000089,13
1,QEQE,-0.568835,0.008521,15
2,QEQQ,-1.016661,0.013544,27
3,QQQQ,-1.642983,0.002901,14
4,QEmQQ,-3.283397,0.001506,13
5,QEmQEm,-4.325549,0.015325,12


In [5]:
import pandas as pd
bootstrap_summary = pd.read_csv(ROOT/"outputs"/"shimizu_uncertainty_layers.csv")
bootstrap_summary

,layer,alpha_median,alpha_2.5%,alpha_97.5%,GmA_2.5%,GmA_97.5%
0,within-state residual bootstrap,2.002883,1.947839,2.056937,2.597118,2.742583
1,between-family structural sensitivity,1.986979,1.841464,2.155739,2.455285,2.874319


## 4. FRET-derived functional noise and static mutual information

In [6]:
mstar, kappa, sigma_A, sigma_R, sigma_m = cra.make_static_functions(cal["alpha"], cal["m0"], cal["GmA"])
for L in [1,10,25]:
    print(L, "uM -> sigma_A =", float(sigma_A(L)), "sigma_R =", float(sigma_R(L)), "sigma_m =", float(sigma_m(L)))
I, Neff = cra.mutual_information(cal["alpha"], cal["m0"], cal["GmA"], 1, 1e4)
print("I(L;m) =", I, "bits")
print("N_eff =", Neff)
print("kappa(100 uM) =", float(kappa(100)))


1 uM -> sigma_A = 0.6782329983125268 sigma_R = 0.06782329983125268 sigma_m = 0.2550202841886844
10 uM -> sigma_A = 0.405 sigma_R = 0.04050000000000001 sigma_m = 0.15228279271782752
25 uM -> sigma_A = 0.363 sigma_R = 0.0363 sigma_m = 0.13649050310264538
I(L;m) = 2.181632308931791 bits
N_eff = 4.536665559535445
kappa(100 uM) = 0.40817182584073974


## 5. Environmental-prior sensitivity

In [7]:
rows=[]
import math, pandas as pd
for lo,hi in cra.PRIOR_WINDOWS_uM:
    Ii, Ni = cra.mutual_information(cal["alpha"], cal["m0"], cal["GmA"], lo, hi)
    rows.append([lo,hi,math.sqrt(lo*hi),Ii,Ni])
prior_df = pd.DataFrame(rows, columns=["Lmin_uM","Lmax_uM","center_uM","I_bits","N_eff"])
prior_df


,Lmin_uM,Lmax_uM,center_uM,I_bits,N_eff
0,0.3,3000.0,30.0,1.922677,3.791260
1,1.0,10000.0,100.0,2.181632,4.536666
2,3.0,30000.0,300.0,2.257435,4.781406
3,10.0,100000.0,1000.0,2.141841,4.413248
4,18.0,180000.0,1800.0,2.014528,4.040484


## 6. Run-scale changes and timescale hierarchy

In [8]:
run_df = cra.run_scale_changes()
times_df = cra.timescale_table(I)
display(run_df)
display(times_df)
print("corner frequency =", 1/(2*math.pi*cra.TAU_M_S), "Hz")


,g_mm^-1,delta_lnL_one_run,delta_lnL_binary_units
0,0.1,0.002,0.002885
1,0.2,0.004,0.005771
2,0.3,0.006,0.008656
3,0.4,0.008,0.011542


,g_mm^-1,tau_env_s,tau_env_min,epsilon_ad
0,0.1,1015.100215,16.918337,0.009851
1,0.2,507.550108,8.459168,0.019702
2,0.3,338.366738,5.639446,0.029554
3,0.4,253.775054,4.229584,0.039405


corner frequency = 0.015915494309189534 Hz


## 7. Dynamic benchmark and boundary-condition robustness

In [9]:
dyn_df = cra.dynamic_summary()
bnd_df = cra.boundary_ratios()
display(dyn_df)
display(bnd_df)


,g_mm^-1,methylation_open_median_bits_s,methylation_open_lo_bits_s,methylation_open_hi_bits_s,kinase_measured_bits_s,rate_scale_ratio_percent
0,0.1,0.000174,0.000152,0.000207,0.0022,7.909091
1,0.2,0.000667,0.000585,0.000797,0.0088,7.579545
2,0.3,0.001420,0.001240,0.001690,0.0198,7.171717
3,0.4,0.002400,0.002110,0.002870,0.0352,6.818182


,g_mm^-1,reflecting_over_open,finite_window_over_open,reinjection_over_open_stress_test
0,0.1,5.658333,3.183333,7.666667
1,0.2,2.637744,2.227766,5.314534
2,0.3,1.777778,1.577982,3.985729
3,0.4,1.193004,1.240651,3.908323


### Interpretation of Table S3 ratios
The revised SI reports reflecting/open and finite-window/open ratios rather than duplicate absolute rates.  
This isolates boundary sensitivity and avoids mixing historical and final susceptibility calibrations.  
Exact calibration-independence of the ratios assumes a common multiplicative recalibration in the weak-SNR regime; a full-spectrum rerun is the definitive validation if that approximation fails.


## 8. Inspect generated outputs

In [10]:
sorted(p.name for p in (ROOT/"outputs").glob("*"))


['Figure_S5_boundary_ratios.png',
 'boundary_ratios_Table_S3.csv',
 'colin_noise_mapping.csv',
 'conditional_methylation_distributions.png',
 'dynamic_rate_scales.png',
 'dynamic_rate_summary.csv',
 'ligand_to_methylation.png',
 'prior_sensitivity.csv',
 'prior_window_sensitivity.png',
 'run_scale_changes.csv',
 'shimizu_endpoint_calibration.png',
 'shimizu_fitted_free_energies.csv',
 'shimizu_uncertainty_layers.csv',
 'summary.json',
 'tau_env_vs_gradient.png',
 'timescale_hierarchy.csv']